# Candy Crush RL Training with OpenAI Gym Standard + TensorBoard

## Workflow
1. Mount Google Drive
2. Clone repository (master branch)
3. Install dependencies
4. Compile C++ extension
5. Test Gym environment
6. Start TensorBoard
7. Run training with Gym standard & TensorBoard logging
8. Save models & logs to Drive

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/candy_crush_rl'
os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}

In [ ]:
# 2. Clone repository (if not already cloned)
REPO_URL = "https://github.com/Pliaustjn/candy_crush_rl.git"
REPO_DIR = os.path.join(WORK_DIR, 'candy_crush_rl')

if not os.path.exists(REPO_DIR):
    !git clone -b master {REPO_URL} {REPO_DIR}
    print(f"Repository cloned to {REPO_DIR}")
else:
    %cd {REPO_DIR}
    !git pull origin master
    print("Repository updated")

%cd {REPO_DIR}

In [ ]:
# 3. Install dependencies
!pip install -q pybind11 numpy torch torchvision gym tensorboard

import pybind11
import torch
import gym
import numpy as np

print(f"PyTorch {torch.__version__}, Gym {gym.__version__}, NumPy {np.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 4. Compile C++ extension
import sys
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))  # allow imports from src

print("Generating C++ sources...")
%run generate_clean_files.py

print("\nCompiling C++ module...")
!python setup.py build_ext --inplace

# Quick test
import candy_crush_cpp
env = candy_crush_cpp.CandyCrushEnv(max_steps=5, target_score=100, seed=42)
board = env.reset()
print(f"C++ env OK: board {len(board)}x{len(board[0])}")

In [ ]:
# 5. Verify Gym environment
from gym_env_wrapper import CandyCrushGymEnv

gym_env = CandyCrushGymEnv(max_steps=5, target_score=100)
obs = gym_env.reset()
print(f"Observation shape: {obs.shape}")
print(f"Action space: {gym_env.action_space}")
print(f"Observation space: {gym_env.observation_space}")

# Run a few random steps
for i in range(3):
    legal = gym_env._get_legal_actions_mask()
    actions = np.where(legal == 1)[0]
    if len(actions) == 0:
        break
    a = np.random.choice(actions)
    obs, rew, done, info = gym_env.step(a)
    print(f"Step {i+1}: action={a}, reward={rew:.1f}, score={info['score']}")
print("Gym interface works correctly!")

In [ ]:
# 6. Start TensorBoard (run in background)
%load_ext tensorboard
%tensorboard --logdir training_logs_*/tensorboard --port 6006

In [ ]:
# 7. Run training with Gym standard + TensorBoard
import sys, os
sys.path.insert(0, os.getcwd())
sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

print("Starting training (OpenAI Gym + TensorBoard)...")
%run train_with_gym_tensorboard.py

In [ ]:
# 8. Save models and logs to Google Drive
import shutil
from datetime import datetime

SAVE_DIR = os.path.join(WORK_DIR, 'saved_models')
os.makedirs(SAVE_DIR, exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')

for f in os.listdir('.'):
    if f.endswith('.pth') or (f.startswith('training_log') and f.endswith('.txt')):
        shutil.copy2(f, os.path.join(SAVE_DIR, f"{ts}_{f}"))
        print(f"Saved: {ts}_{f}")

# Also save TensorBoard logs
for d in os.listdir('.'):
    if d.startswith('training_logs_'):
        dest = os.path.join(SAVE_DIR, f"{ts}_{d}")
        if not os.path.exists(dest):
            shutil.copytree(d, dest)
            print(f"Saved TensorBoard logs: {dest}")

print(f"\nAll artifacts saved to {SAVE_DIR}")